# VLM highlight scores for FPVSum (Track 04, dev experiment)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**, add your private FPVSum dataset (the folder of `.mp4` files).

**Run All.** ~1–2 h. Output: `/kaggle/working/vlm_scores.zip` (one `.npz` per video). Download it from the Output tab.

For each 4-second window (4 frames at 1 fps, stride 2 s) Qwen2.5-VL answers *how much would you include this moment in a highlight reel* with one digit 0–9.
The score is the **expected digit under the model's next-token distribution**, which ranks windows more smoothly than the text answer.
No ground truth, file names or categories are used — every video gets the same generic prompts.
The notebook is resumable: re-running skips videos that are already scored.

In [ ]:
!pip install -q "transformers==4.51.3" "accelerate>=0.34"
import torch, transformers
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import glob, os, time, zipfile, math
import numpy as np, cv2, torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

VIDEOS = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
OUT = "/kaggle/working/vlm_scores"; os.makedirs(OUT, exist_ok=True)
print(len(VIDEOS), "videos found");  assert VIDEOS, "attach the FPVSum dataset"

WIN, STRIDE = 4, 2          # window of 4 frames @ 1 fps, stride 2 s
SHORT, LONG = 252, 448      # frame size (multiples of 28)
BATCH = 8
PROMPTS = [
    "These are 4 consecutive seconds from a first-person (head- or body-mounted camera) video. "
    "If you were cutting a short highlight reel of the whole video, how much would you want to include this moment? "
    "Answer with a single digit from 0 (boring, skip it) to 9 (must include).",
    "These are 4 consecutive seconds from a first-person video. "
    "How exciting, eventful or visually interesting is what happens here? "
    "Answer with a single digit from 0 (nothing happens) to 9 (the most exciting moment).",
]

In [ ]:
def load_model(name, dtype):
    mm = {i: "13GiB" for i in range(torch.cuda.device_count())}
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        name, torch_dtype=dtype, device_map="auto", max_memory=mm, attn_implementation="sdpa").eval()
    proc = AutoProcessor.from_pretrained(name)
    proc.tokenizer.padding_side = "left"
    return model, proc

def digit_ids(tok):
    ids = [tok.encode(str(d), add_special_tokens=False) for d in range(10)]
    assert all(len(i) == 1 for i in ids), ids
    return [i[0] for i in ids]

@torch.no_grad()
def score_batch(model, proc, clips, prompt):
    msgs = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": prompt}]}]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = proc(text=[text] * len(clips), videos=clips, fps=1.0, padding=True, return_tensors="pt")
    inp = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inp.items()}
    logits = model(**inp).logits[:, -1, :].float()
    p_all = torch.softmax(logits, -1)
    p = p_all[:, DIG]
    mass = p.sum(-1)
    p = p / mass.clamp_min(1e-9)[:, None]
    return p.cpu().numpy(), mass.cpu().numpy()

def sane(model, proc):
    clip = [Image.fromarray(np.random.randint(0, 255, (SHORT, LONG, 3), np.uint8)) for _ in range(WIN)]
    p, m = score_batch(model, proc, [clip, clip], PROMPTS[0])
    print("sanity: digit mass", m, "probs ok:", np.isfinite(p).all())
    return np.isfinite(p).all() and m.min() > 0.5

# 7B in fp16 split over the two T4s; fall back to 3B in fp32 if fp16 overflows (T4 has no bf16)
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
model, proc = load_model(MODEL, torch.float16)
DIG = digit_ids(proc.tokenizer)
if not sane(model, proc):
    del model; torch.cuda.empty_cache()
    MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
    model, proc = load_model(MODEL, torch.float32)
    assert sane(model, proc), "model output is not usable"
print("using", MODEL)

In [ ]:
def frames_1fps(path):
    """Decode one frame per second (at t = k + 0.5 s), resized to SHORT x LONG keeping orientation."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur = n / fps
    want = [int(round((k + 0.5) * fps)) for k in range(int(math.floor(dur)))]
    out, i, j = [], 0, 0
    while j < len(want):
        ok = cap.grab()
        if not ok:
            break
        if i == want[j]:
            ok, f = cap.retrieve()
            if ok:
                h, w = f.shape[:2]
                size = (LONG, SHORT) if w >= h else (SHORT, LONG)
                out.append(Image.fromarray(cv2.cvtColor(cv2.resize(f, size, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)))
            j += 1
        i += 1
    cap.release()
    return out, dur

t0 = time.time()
for vi, path in enumerate(VIDEOS):
    vid = os.path.splitext(os.path.basename(path))[0]
    dst = os.path.join(OUT, vid + ".npz")
    if os.path.exists(dst):
        continue
    frames, dur = frames_1fps(path)
    if len(frames) < WIN:
        frames = frames + [frames[-1]] * (WIN - len(frames))
    starts = list(range(0, max(1, len(frames) - WIN + 1), STRIDE))
    if starts[-1] + WIN < len(frames):
        starts.append(len(frames) - WIN)
    clips = [frames[s:s + WIN] for s in starts]
    P = np.zeros((len(clips), len(PROMPTS), 10), np.float32); M = np.zeros((len(clips), len(PROMPTS)), np.float32)
    for q, prompt in enumerate(PROMPTS):
        for b in range(0, len(clips), BATCH):
            p, m = score_batch(model, proc, clips[b:b + BATCH], prompt)
            P[b:b + len(p), q] = p; M[b:b + len(p), q] = m
    score = (P * np.arange(10)).sum(-1)                    # expected digit, windows x prompts
    np.savez_compressed(dst, t=np.array(starts, np.float32) + WIN / 2, score=score, probs=P, mass=M,
                        duration=dur, model=MODEL)
    print(f"{vi + 1}/{len(VIDEOS)} {vid}: {len(clips)} windows, mean score {score.mean(0).round(2)}, "
          f"digit mass {M.mean():.2f}  [{(time.time() - t0) / 60:.1f} min]", flush=True)

In [ ]:
with zipfile.ZipFile("/kaggle/working/vlm_scores.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(OUT, "*.npz"))):
        z.write(f, "vlm_scores/" + os.path.basename(f))
print(len(glob.glob(os.path.join(OUT, "*.npz"))), "videos scored -> /kaggle/working/vlm_scores.zip")